# 06_collect — 팀 결과 통합 점검과 보고서 표·그림 (담당: core)

`outputs/{logit, xgb, xgb_mono}/`에 올라온 결과를 읽어 두 가지를 합니다.

1. **점검:** 세 모형이 같은 버전(config·lib_prep 해시), 같은 열(contract), 같은 B, 같은 출력 형식으로 돌았는지 확인합니다. 하나라도 다르면 표를 만들기 전에 멈추고 무엇이 다른지 알려 줍니다.
2. **보고서용:** 표 4(`report/tables/t4_validation_summary.csv`), 그림 2·3·4(`report/figures/fig2~4`)를 만듭니다.

아직 올라오지 않은 모형은 "없음"으로 표시하고 있는 모형만으로 만듭니다. 그림 4(ΔSR)는 ②가 있어야 그립니다.
B=100 본 실행 결과가 모두 모이면 이 노트북만 다시 실행하면 됩니다.

In [1]:
SKIP_LIB_TESTS = True
%run -i ./00_config.ipynb
%run -i ./lib_prep.ipynb
%run -i ./lib_eval.ipynb
import json, joblib
import matplotlib.pyplot as plt
from matplotlib import font_manager
for name in ("AppleGothic", "Apple SD Gothic Neo", "Malgun Gothic", "NanumGothic", "Noto Sans CJK KR"):
    if name in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = name; break
plt.rcParams["axes.unicode_minus"] = False
MODELS = {"logit": "① 로지스틱", "xgb": "② XGBoost", "xgb_mono": "③ 단조 XGBoost"}
TAB_DIR = REPORT_DIR / "tables"; FIG_DIR = REPORT_DIR / "figures"
TAB_DIR.mkdir(parents=True, exist_ok=True); FIG_DIR.mkdir(parents=True, exist_ok=True)
NOW = dict(config_hash=config_fingerprint(), lib_prep_hash=LIB_PREP_HASH)
print("지금 버전:", NOW)

ModuleNotFoundError: No module named 'nbformat'

In [ ]:
# 1. 점검 (모형마다 PASS/FAIL, 없는 모형은 건너뜀)
REQ_RES = ["b", "a_star", "tau_star", "sr_max", "auc", "brier"]
REQ_CURVE = ["b", "rate", "sr", "tau"]
REQ_SUMMARY = ["model", "a_star", "tau_star_median", "columns", "config_hash", "lib_prep_hash"]
cols_contract = {"logit": json.load(open(CONTRACT_DIR / "columns_logit.json")),
                 "xgb": json.load(open(CONTRACT_DIR / "columns_xgb.json"))}
cols_contract["xgb_mono"] = cols_contract["xgb"]

have, report = {}, []
for m, label in MODELS.items():
    d = OUT_DIR / m
    if not (d / "summary.json").exists():
        report.append((label, "없음", "outputs/%s/summary.json 없음" % m)); continue
    s = json.load(open(d / "summary.json"))
    res = pd.read_csv(d / "bootstrap_results.csv"); cur = pd.read_csv(d / "bootstrap_curves.csv")
    n_models = len(list((d / "models_b").glob("model_b*.joblib"))) if (d / "models_b").exists() else 0
    checks = {
        "summary 공통 키": all(k in s for k in REQ_SUMMARY),
        "config 해시 = 지금": s.get("config_hash") == NOW["config_hash"],
        "lib_prep 해시 = 지금": s.get("lib_prep_hash") == NOW["lib_prep_hash"],
        "열 = contract": s.get("columns") == cols_contract[m],
        "결과 열 형식": list(res.columns) == REQ_RES,
        "곡선 열 형식": list(cur.columns) == REQ_CURVE,
        "회차 수 = B": len(res) == B,
        "보관 모형 수 = 회차 수": n_models == len(res),
        "최종 모형 있음": (d / "final_model.joblib").exists(),
        "a* = 회차 중앙값": np.isclose(s.get("a_star", np.nan), res["a_star"].median()),
    }
    bad = [k for k, v in checks.items() if not v]
    report.append((label, "PASS" if not bad else "FAIL", ", ".join(bad) if bad else f"B={len(res)}"))
    have[m] = dict(summary=s, res=res, curves=cur)
chk = pd.DataFrame(report, columns=["모형", "판정", "비고"])
print(chk.to_string(index=False))
FAILED = chk["판정"].eq("FAIL").any()
if FAILED:
    print("\nFAIL이 있으면 표와 그림은 '시험용'으로만 만듭니다. 담당자에게 비고의 항목을 고쳐 달라고 요청하세요.")

In [ ]:
# 2. 표 4: Dev OOB 요약 (회차 B개)
rows = []
for m, label in MODELS.items():
    if m not in have: continue
    r = have[m]["res"]
    rows.append({"모형": label, "B": len(r), "a* 중앙값": r["a_star"].median(),
                 "a* 범위": f"{r['a_star'].min():.2f}~{r['a_star'].max():.2f}",
                 "SR* 평균": r["sr_max"].mean(), "AUC 평균": r["auc"].mean(), "Brier 평균": r["brier"].mean()})
t4 = pd.DataFrame(rows).round(4)
tag = "_시험용" if FAILED else ""
t4.to_csv(TAB_DIR / f"t4_validation_summary{tag}.csv", index=False, encoding="utf-8-sig")
print(t4.to_string(index=False))

In [ ]:
# 3. 그림 2 (승인율별 SR 평균 ±1SD)와 그림 3 (회차별 a*_b 분포)
fig, ax = plt.subplots(figsize=(6.5, 3.4))
for i, (m, label) in enumerate(MODELS.items()):
    if m not in have: continue
    g = have[m]["curves"].groupby("rate")["sr"]; mu, sd = g.mean(), g.std(ddof=1).fillna(0)
    ax.plot(mu.index, mu.values, label=label, color=f"C{i}")
    ax.fill_between(mu.index, mu - sd, mu + sd, alpha=0.15, color=f"C{i}")
    ax.axvline(have[m]["res"]["a_star"].median(), ls="--", lw=0.8, color=f"C{i}")
ax.set_xlabel("승인율 a"); ax.set_ylabel("OOB SR (거절=0)"); ax.legend(frameon=False, fontsize=8)
for s_ in ("top", "right"): ax.spines[s_].set_visible(False)
fig.tight_layout(); fig.savefig(FIG_DIR / f"fig2_oob_sr_curves{tag}.png", dpi=300)

fig, ax = plt.subplots(figsize=(6.5, 2.8))
for i, (m, label) in enumerate(MODELS.items()):
    if m not in have: continue
    ax.hist(have[m]["res"]["a_star"], bins=np.arange(0.30, 0.71, 0.01), histtype="step", lw=1.6, label=label, color=f"C{i}")
ax.set_xlabel("회차별 최적 승인율 a*_b"); ax.set_ylabel("회차 수"); ax.legend(frameon=False, fontsize=8)
for s_ in ("top", "right"): ax.spines[s_].set_visible(False)
fig.tight_layout(); fig.savefig(FIG_DIR / f"fig3_astar_dist{tag}.png", dpi=300)
plt.show()

In [ ]:
# 4. 그림 4: 회차별 ΔSR (②의 a*_b 지점 기준, ② − 다른 모형). ②가 없으면 건너뜀
if "xgb" not in have:
    print("② 결과가 없어 그림 4는 건너뜀")
else:
    ref = have["xgb"]["curves"]; a_ref = have["xgb"]["res"].set_index("b")["a_star"]
    fig, ax = plt.subplots(figsize=(6.5, 3.2)); rows4 = []
    for i, m, label in ((2, "xgb_mono", "② − ③ (단조 제약 비용)"), (0, "logit", "② − ① (ML 이득)")):
        if m not in have: continue
        if m == "xgb_mono" and "DELTA" in PENDING:   # [미정] δ 확정 전에는 ②−③을 보지 않는다(판정 기준이 결과에 끌려가지 않게)
            print("δ 미정: ②−③ 곡선은 δ를 00_config에 확정한 뒤 다시 실행하면 그립니다"); continue
        mm = ref.merge(have[m]["curves"], on=["b", "rate"], suffixes=("_r", "_n"))
        mm["d"] = mm["sr_r"] - mm["sr_n"]
        mean = mm.groupby("rate")["d"].mean()
        ax.plot(mean.index, mean.values, label=label, color=f"C{i}")
        at = mm[np.isclose(mm["rate"], mm["b"].map(a_ref))]
        rows4.append({"비교": label, "ΔSR 중앙값(②의 a*_b)": at["d"].median(),
                      "최소": at["d"].min(), "최대": at["d"].max(), "음수 회차": int((at["d"] < 0).sum())})
    ax.axhline(0, color="k", lw=0.8); ax.set_xlabel("승인율 a"); ax.set_ylabel("ΔSR (OOB, 회차 평균)")
    ax.legend(frameon=False, fontsize=8)
    for s_ in ("top", "right"): ax.spines[s_].set_visible(False)
    fig.tight_layout(); fig.savefig(FIG_DIR / f"fig4_oob_dsr{tag}.png", dpi=300); plt.show()
    print(pd.DataFrame(rows4).round(4).to_string(index=False))

In [ ]:
# 5. 부록 C: ② 최종 모형의 PDP (부호 ±1 변수 20개, Dev 무작위 5,000건, 원 단위 5~95% 분위수 격자)
f = OUT_DIR / "xgb" / "final_model.joblib"
if not f.exists():
    print("② 최종 모형 없음 → 부록 C 건너뜀")
else:
    prep, model = joblib.load(f)
    Xs = pd.read_parquet(DATA_DIR / "dev_X_raw.parquet").sample(5_000, random_state=SEED)
    cols = list(SIGN_OF); nr = int(np.ceil(len(cols) / 5))
    fig, axs = plt.subplots(nr, 5, figsize=(6.5, 1.5 * nr), squeeze=False); rows = []
    for ax, c in zip(axs.flat, cols):
        s_ = pdp_1d(prep, model, Xs, c)
        rho = pd.Series(s_.values).corr(pd.Series(s_.index), method="spearman") if len(s_) > 2 else np.nan
        ax.plot(s_.index, s_.values, color="C1" if SIGN_OF[c] > 0 else "C0", lw=1.2)
        ax.set_title(f"{c} ({SIGN_OF[c]:+d})", fontsize=5.5); ax.tick_params(labelsize=4.5); tidy(ax)
        rows.append(dict(변수=c, 부호=SIGN_OF[c], pdp_rho=rho, 변화폭=s_.iloc[-1] - s_.iloc[0]))
    for ax in list(axs.flat)[len(cols):]: ax.axis("off")
    fig.tight_layout(); fig.savefig(FIG_DIR / f"figA2_pdp{tag}.png", dpi=300); plt.show()
    pdt = pd.DataFrame(rows)
    pdt["판정"] = np.where(pdt["변화폭"].abs() < 1e-3, "평평(|변화폭|<0.001)", np.where(np.sign(pdt["변화폭"]) == pdt["부호"], "일치", "반대"))   # [미정·기본값] 평평 기준 0.001
    pdt.round(4).to_csv(TAB_DIR / f"tA3_pdp_check{tag}.csv", index=False, encoding="utf-8-sig")
    print(pdt.round(4).to_string(index=False))